In [1]:
import pandas as pd

# CSV-Datei einlesen
# Quelle: https://www.bbsr.bund.de/BBSR/DE/forschung/raumbeobachtung/Raumabgrenzungen/downloads/download-referenzen.html
df = pd.read_excel("../data/raw/raumgliederungen-referenzen-2023.xlsx", skiprows=1, sheet_name="Kreisreferenz")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 66 columns):
 #   Column                                                               Non-Null Count  Dtype 
---  ------                                                               --------------  ----- 
 0   Kreise (2023) Kennziffer                                             400 non-null    int64 
 1   Kreise (2023) Name                                                   400 non-null    object
 2   Arbeitsagenturbezirke (2023) Kennziffer                              400 non-null    int64 
 3   Arbeitsagenturbezirke (2023) Name                                    400 non-null    object
 4   Arbeitsmarktregionen (2023) Kennziffer                               400 non-null    int64 
 5   Arbeitsmarktregionen (2023) Name                                     400 non-null    object
 6   Braunkohlereviere (auch nicht förderfähige) (2023) Kennziffer        400 non-null    int64 
 7   Braunkohlereviere

In [2]:
df=df[["Kreise (2023) Name", "Raumordnungsregionen (2023) Name"]]
df=df.rename(columns={"Kreise (2023) Name": "Name", "Raumordnungsregionen (2023) Name": "ROR"})
df2 = pd.read_csv("../data/processed/hilfen_2024.csv")

import re
names = (
    df2["Name"]
    .dropna()
    .astype(str)
    .str.lower()
    .apply(re.escape)  
)

pattern = r"\b(" + "|".join(names) + r")\b"

df = df.loc[
    df["Name"]
      .astype(str)
      .str.lower()
      .str.contains(pattern, na=False, regex=True)
]



C:\Users\inaba\AppData\Local\Temp\ipykernel_3344\3887436067.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, na=False, regex=True)


In [3]:
from ipynb.fs.defs.functions import clean_and_sort, validate_df

df = clean_and_sort(df, "ROR")

In [4]:
#df = df.merge(df2, on=["ROR"], how="left")
set_hilfen = set(df2["Name"])
set_ror = set(df["Name"])
no_match = set_ror.symmetric_difference(set_hilfen)
print(no_match)

{'Solingen, Klingenstadt', 'Solingen', 'Hagen', 'Hagen, Stadt der FernUniversität'}


In [5]:
mapping = {'Hagen, Stadt der FernUniversität':'Hagen', 'Solingen, Klingenstadt': 'Solingen'}
df["Name"] = df["Name"].replace(mapping)

print(df, df.info())
df.to_csv("../data/processed/ror.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Name    53 non-null     object
 1   ROR     53 non-null     object
dtypes: object(2)
memory usage: 980.0+ bytes
                          Name             ROR
0                   Düsseldorf      Düsseldorf
1                     Duisburg  Duisburg/Essen
2                        Essen  Duisburg/Essen
3                      Krefeld      Düsseldorf
4              Mönchengladbach      Düsseldorf
5          Mülheim an der Ruhr  Duisburg/Essen
6                   Oberhausen  Duisburg/Essen
7                    Remscheid      Düsseldorf
8                     Solingen      Düsseldorf
9                    Wuppertal      Düsseldorf
10                       Kleve  Duisburg/Essen
11                    Mettmann      Düsseldorf
12           Rhein-Kreis Neuss      Düsseldorf
13                     Viersen      Düsseldorf
14  